In [1]:
from unsloth import FastLanguageModel, is_bfloat16_supported
import json
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments, TextStreamer

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/flima/workspace/AI/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Unsloth: Failed to patch Gemma3ForConditionalGeneration.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
# Test GPU is available
import torch
print(torch.cuda.is_available()) # Should be true
print(torch.cuda.get_device_name(0)) # If available

True
NVIDIA GeForce RTX 3060


In [ ]:
DATA_PATH = "./data/trn.json" # Base dataset on readme
OUTPUT_PATH_DATASET = "./data/formatted_data.json"

max_seq_length = 2048
dtype = None
load_in_4bit = True
fourbit_models = [
    "unsloth/gemma-3-1b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-4b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-27b-it-unsloth-bnb-4bit",
]

In [ ]:
def format_dataset_into_model_input(data):
    instructions = []
    inputs = []
    outputs = []

    for obj in data:
        instructions.append("DESCRIBE ABOUT THE PRODUCT.")
        inputs.append(obj.get("title", ""))
        outputs.append(obj.get("content", ""))
    
    final_output = {
        "instruction": instructions,
        "input": inputs,
        "output": outputs
    }

    with open(OUTPUT_PATH_DATASET, 'w') as output_file:
        json.dump(final_output, output_file, indent=4)
    
    print(f"Dataset saved in {OUTPUT_PATH_DATASET}")


In [ ]:
# Open file without formatation
with open(DATA_PATH, "r") as f:
    data = [json.loads(line) for line in f]

format_dataset_into_model_input(data)

In [4]:
# Loading model in GPU memory
model, processor = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-3-1b-it-unsloth-bnb-4bit",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True
)

==((====))==  Unsloth 2025.3.19: Fast Gemma3 patching. Transformers: 4.51.3.
   \\   /|    NVIDIA GeForce RTX 3060. Num GPUs = 1. Max memory: 11.761 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu121. CUDA: 8.6. CUDA Toolkit: 12.1. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


In [5]:
model.gradient_checkpointing_enable()

In [6]:
# Get tokenizer from processor
tokenizer = processor

In [7]:
# Check tokenizer
print(type(processor))

<class 'transformers.models.gemma.tokenization_gemma_fast.GemmaTokenizerFast'>


In [8]:
model = FastLanguageModel.get_peft_model(
    model,
    finetune_vision_layers = False,
    finetune_language_layers = True,
    finetune_attention_modules = True,
    finetune_mlp_modules = True,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407
)

Unsloth: Making `model.base_model.model.model` require gradients


In [9]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

In [15]:
# Running before the fine tunning
inputs = tokenizer(
    [
        alpaca_prompt.format(
            "DESCRIBE ABOUT THE PRODUCT.",
            "The Book of Revelation",
            "",
        )
    ], return_tensors = "pt").to("cuda")

text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128)

<bos>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
DESCRIBE ABOUT THE PRODUCT.

### Input:
The Book of Revelation

### Response:
The Book of Revelation is a complex and enigmatic text originating from the Second Temple period of Judaism. It is a collection of apocalyptic visions and symbolic representations of divine judgment, spiritual warfare, and the ultimate triumph of God. This work explores themes of persecution, exile, and salvation. The imagery is predominantly symbolic and often challenges conventional understanding of the world. It's often interpreted as a prophetic message from the end times.

**Here's a more detailed breakdown of its key aspects:**

* **Origin and Time Period:** It emerged during the Second Temple period (roughly 580-35 BCE), a time of intense social


In [16]:
# Formats the dataset into a specific prompt strucutre for fine-tunning a language model.
EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []

    for instruction, input, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text": texts }
pass

dataset = load_dataset("json", data_files = OUTPUT_PATH_DATASET, split = "train")
dataset = dataset.map(formatting_prompts_func, batched = True)

Generating train split: 2248619 examples [00:19, 115391.94 examples/s]
Map: 100%|██████████| 2248619/2248619 [00:51<00:00, 43709.81 examples/s] 


In [17]:
import os
os.environ["TORCHINDUCTOR_DISABLE"] = "1"

In [18]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4, # Use GA to mimic batch size!
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs"
    )
)

Map (num_proc=2): 100%|██████████| 2248619/2248619 [07:01<00:00, 5334.49 examples/s]


In [19]:
# Clean cache from cuda
import torch
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

In [20]:
# Train the model with dataset
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,248,619 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 13,045,760/1,000,000,000 (1.30% trained)
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
1,4.125100
2,4.193700
3,4.390200
4,3.411000
5,3.875000
6,3.541800
7,3.175000
8,2.944700
9,2.972100
10,3.055500


Unsloth: Will smartly offload gradients to save VRAM!


In [34]:
inputs = tokenizer(
    [
        alpaca_prompt.format(
            "DESCRIBE ABOUT THE PRODUCT.",
            "The Book of Revelation",
            "",
        )
    ], return_tensors = "pt").to("cuda")

text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128)

<bos>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
DESCRIBE ABOUT THE PRODUCT.

### Input:
The Book of Revelation

### Response:
The Book of Revelation. A second edition of the first edition of the Book of Revelation will provide clarity and understanding for those with an established theological background, or those who are new to the book. A text for all who seek to understand the end of days. The message of the Book of Revelation is one of hope in the face of despair, assurance that God is in control, and hope for the world for the end of the times.<end_of_turn>


In [26]:
# Any prompt only to test
prompt = "Tell me only one joke about computers."

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens = 128)
print(tokenizer.decode(outputs[0]))

<bos>Tell me only one joke about computers.

###A computer who is worried about the price of the next computer in line said, "I can't believe it is expensive!"###<end_of_turn>
